# 🔵 AI Morse Code Telegraph Translator

Real-time Morse code decoder with **keyboard** and **microphone** input modes,
ML-powered tap detection, and text-to-speech output.

| Mode | How to input |
|------|-------------|
| ⌨️ **Keyboard** | Hold `Space` — short press = `.`  long press = `−` |
| 🎙️ **Microphone** | Tap the table — classifier filters real taps from noise |

**Workflow:**
1. Run **Setup** cells (imports → dictionary → parameters → functions)
2. **Keyboard mode** — run the Keyboard Listener cell directly
3. **Mic mode** — run Calibration first, then the Mic Listener cell

In [ ]:
import sys
import os
import time
import threading
import queue
import pickle
import subprocess
import platform

import numpy as np
from IPython.display import display, HTML
import keyboard

try:
    import sounddevice as sd
    MIC_AVAILABLE = True
except Exception as e:
    MIC_AVAILABLE = False
    print(f"⚠️  sounddevice not available — keyboard mode only. ({e})")

try:
    import pyttsx3
    TTS_AVAILABLE = True
except Exception:
    TTS_AVAILABLE = False

from sklearn.ensemble import RandomForestClassifier

print("✅ Libraries loaded.")
print(f"   🎙  Microphone mode : {'available' if MIC_AVAILABLE else 'unavailable'}")
print(f"   🔊  TTS (pyttsx3)   : {'available (fallback)' if TTS_AVAILABLE else 'not installed — using system TTS'}")

## 📡 Morse Code Dictionary

In [17]:
morse_dict = {
    # Letters
    ".-": "A", "-...": "B", "-.-.": "C", "-..": "D", ".": "E",
    "..-.": "F", "--.": "G", "....": "H", "..": "I", ".---": "J",
    "-.-": "K", ".-..": "L", "--": "M", "-.": "N", "---": "O",
    ".--.": "P", "--.-": "Q", ".-.": "R", "...": "S", "-": "T",
    "..-": "U", "...-": "V", ".--": "W", "-..-": "X", "-.--": "Y",
    "--..": "Z",

    # Numbers
    "-----": "0", ".----": "1", "..---": "2", "...--": "3",
    "....-": "4", ".....": "5", "-....": "6", "--...": "7",
    "---..": "8", "----.": "9",

    # Common punctuation
    ".-.-.-": ".", "--..--": ",", "..--..": "?",
    ".----.": "'", "-.-.--": "!", "-..-.": "/",
    "-.--.": "(", "-.--.-": ")", ".-...": "&",
    "---...": ":", "-.-.-.": ";", "-...-": "=",
    ".-.-.": "+", "-....-": "-", "..--.-": "_",
    ".-..-.": "\"", "...-..-": "$", ".--.-.": "@"
}

## ⚙️ Configuration

Tune these values if signals are missed or noise triggers false detections.

In [ ]:
# ── Signal timing ─────────────────────────────────────────────
DOT_DURATION  = 0.15   # Max seconds for a dot (longer press → dash)
LETTER_GAP    = 0.40   # Silence (s) before a letter is finalised
WORD_GAP      = 1.00   # Silence (s) before a word space is inserted

# ── Microphone detection ───────────────────────────────────────
GAIN          = 100    # Software amplification applied to mic input
SPIKE_FACTOR  = 3.0   # Tap must be this many × louder than the noise floor
NOISE_SMOOTH  = 0.997  # Noise-floor adaptation rate (higher = more stable)
SAMPLE_RATE   = 44100  # Audio sample rate in Hz

# ── Text-to-Speech ─────────────────────────────────────────────
TTS_ENABLED   = True   # Set False to disable spoken output

## 🔧 Core Functions

**Signal pipeline:**

```
Input (Space / Mic) → Duration Measure → Dot/Dash Classify
      → Symbol Buffer → Gap Detect → Morse Lookup → Text + TTS
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# Signal helpers
# ─────────────────────────────────────────────────────────────

def measure_duration(start_time):
    """Return elapsed seconds since start_time."""
    return time.time() - start_time


def classify_signal(duration, dot_limit=DOT_DURATION):
    """Return '.' for a short press, '-' for a long press."""
    return "." if duration < dot_limit else "-"


def decode_morse(symbol_buffer, dictionary=None):
    """Look up a dot/dash sequence in the Morse dictionary.
    Returns the matched character, or '?' if unknown."""
    if dictionary is None:
        dictionary = morse_dict
    return dictionary.get(symbol_buffer, "?")


def extract_features(block):
    """Extract 6 audio features from one frame for tap vs noise classification."""
    block          = block.flatten()
    rms            = np.sqrt(np.mean(block ** 2)) + 1e-10
    peak           = np.max(np.abs(block))
    crest_factor   = peak / rms
    zcr            = np.sum(np.abs(np.diff(np.sign(block)))) / (2 * len(block))
    fft_mag        = np.abs(np.fft.rfft(block))
    freqs          = np.fft.rfftfreq(len(block), d=1.0 / SAMPLE_RATE)
    spectral_cent  = np.sum(freqs * fft_mag) / (np.sum(fft_mag) + 1e-10)
    q              = len(block) // 4
    attack         = np.sqrt(np.mean(block[:q] ** 2)) / rms if q > 0 else 1.0
    return np.array([rms, peak, crest_factor, zcr, spectral_cent, attack])


# ─────────────────────────────────────────────────────────────
# Text-to-Speech
# ─────────────────────────────────────────────────────────────

def speak_text(text):
    """Speak text non-blocking using the best available engine for this OS."""
    if not TTS_ENABLED:
        return
    try:
        if platform.system() == "Darwin":            # macOS: built-in 'say' command
            subprocess.Popen(["say", text])
        elif platform.system() == "Windows":          # Windows: pyttsx3
            if TTS_AVAILABLE:
                def _speak():
                    engine = pyttsx3.init()
                    engine.say(text)
                    engine.runAndWait()
                threading.Thread(target=_speak, daemon=True).start()
        else:                                         # Linux: espeak → pyttsx3 fallback
            try:
                subprocess.Popen(["espeak", text])
            except FileNotFoundError:
                if TTS_AVAILABLE:
                    def _speak():
                        engine = pyttsx3.init()
                        engine.say(text)
                        engine.runAndWait()
                    threading.Thread(target=_speak, daemon=True).start()
    except Exception:
        pass   # TTS is non-critical; never crash the decoder


# ─────────────────────────────────────────────────────────────
# HTML dashboard renderers
# ─────────────────────────────────────────────────────────────

_DARK = "background:#1e1e1e;color:#d4d4d4;padding:14px;border-radius:8px;"


def render_keyboard_html(raw_log, text, is_pressing):
    bar_color  = "#4caf50" if is_pressing else "#555"
    key_symbol = "█ SPACE" if is_pressing else "▁ space"
    status_txt = "TRANSMITTING" if is_pressing else "ready"
    return HTML(
        f"<div style='font-family:monospace;font-size:14px;line-height:1.8;{_DARK}'>"
        f"<b>⌨️  Key:</b> "
        f"<span style='background:{bar_color};color:white;padding:2px 10px;"
        f"border-radius:4px;font-weight:bold;'>{key_symbol}</span> "
        f"<span style='color:#999;'>{status_txt}</span><br><br>"
        f"<b>Signals:</b> {raw_log if raw_log else '<i style=\"color:#555;\">hold Space to begin...</i>'}<br><br>"
        f"<b style='font-size:16px;'>Decoded:</b> "
        f"<span style='font-size:22px;color:#4ec9b0;'>"
        f"{text if text else '<i style=\"color:#555;\">–</i>'}</span>"
        f"</div>"
    )


def render_mic_html(raw_log, text, vol, threshold, floor, label):
    bar_max   = 50
    cap       = max(threshold * 3, 0.01)
    scale     = min(vol / cap, 1.0)
    filled    = int(scale * bar_max)
    is_tap    = label.startswith("TAP")
    bar_color = "#4caf50" if is_tap else "#666"
    bar       = (
        f"<span style='color:{bar_color};'>{'█' * filled}</span>"
        f"<span style='color:#333;'>{'░' * (bar_max - filled)}</span>"
    )
    if is_tap:
        status = f"🟢 {label}"
    elif label.startswith("rejected"):
        status = f"🟡 {label}"
    elif label == "detecting…":
        status = "🔵 detecting…"
    else:
        status = "⚫ quiet"
    return HTML(
        f"<div style='font-family:monospace;font-size:14px;line-height:1.8;{_DARK}'>"
        f"<b>🎤 Mic:</b> |{bar}| {status} "
        f"<span style='color:#555;'>vol={vol:.4f} thresh={threshold:.4f} floor={floor:.4f}</span><br><br>"
        f"<b>Signals:</b> {raw_log if raw_log else '<i style=\"color:#555;\">tap the table to begin...</i>'}<br><br>"
        f"<b style='font-size:16px;'>Decoded:</b> "
        f"<span style='font-size:22px;color:#4ec9b0;'>"
        f"{text if text else '<i style=\"color:#555;\">–</i>'}</span>"
        f"</div>"
    )


print("✅ Functions loaded.")

## 🎙️ Mic Calibration — Train the Tap Detector

> **Run once before using Mic Mode.** The model is saved to `tap_model.pkl` and reloaded automatically.

1. Run the cell below
2. **First 5 s** — sit still, don't tap (records background noise)
3. **Next 5 s** — tap the table repeatedly and clearly

The Random Forest classifier learns the difference between real taps and ambient noise.

In [ ]:
# ── Step 1: Record background noise ──────────────────────────
print("🔇  Recording 5 s of BACKGROUND NOISE — sit still, do NOT tap...\n")
bg_rec = sd.rec(int(5 * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
bg_rec = bg_rec * GAIN
print("   ✅ Background recorded.\n")

# ── Step 2: Record taps ───────────────────────────────────────
print("🔊  Recording 5 s of TAPS — tap the table repeatedly NOW!\n")
tap_rec = sd.rec(int(5 * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
tap_rec = tap_rec * GAIN
print("   ✅ Taps recorded.\n")

# ── Step 3: Extract features ──────────────────────────────────
blocksize = 1024
X, y = [], []

# Background frames → label 0
for i in range(0, len(bg_rec) - blocksize, blocksize):
    X.append(extract_features(bg_rec[i:i + blocksize]))
    y.append(0)

# Determine noise floor RMS
bg_rms = np.mean([
    np.sqrt(np.mean(bg_rec[i:i + blocksize] ** 2))
    for i in range(0, len(bg_rec) - blocksize, blocksize)
])

# Tap frames → label 1 if loud, else 0 (quiet moments within the tap recording)
for i in range(0, len(tap_rec) - blocksize, blocksize):
    block     = tap_rec[i:i + blocksize]
    block_rms = np.sqrt(np.mean(block ** 2))
    X.append(extract_features(block))
    y.append(1 if block_rms > bg_rms * 2 else 0)

X = np.array(X)
y = np.array(y)
print(f"   Training samples: {len(y)}  ({(y == 0).sum()} noise | {(y == 1).sum()} tap)\n")

# ── Step 4: Train the Random Forest classifier ────────────────
tap_model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
tap_model.fit(X, y)
print(f"   Training accuracy: {tap_model.score(X, y):.1%}")

# ── Step 5: Save model ────────────────────────────────────────
MODEL_PATH = "tap_model.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(tap_model, f)
print(f"\n✅ Tap detector saved → {MODEL_PATH}")
print("   You can now run the Mic Listener cell.")

## ⌨️ Keyboard Mode

Use your keyboard as a telegraph key — **no microphone or calibration needed**.

| Action | Meaning |
|--------|---------|
| Short `Space` press | `.` dot |
| Long `Space` press (> 0.15 s) | `−` dash |
| Pause ≥ 0.4 s | End of letter |
| Pause ≥ 1.0 s | End of word |
| `ESC` | Stop |

In [ ]:
# ── Shared state ──────────────────────────────────────────────
stop_event   = threading.Event()
output_q     = queue.Queue()
decoded_text = ""


# ── Keyboard input thread ─────────────────────────────────────
def _keyboard_input(output_q, stop_event):
    """Polls the Space bar; converts press/release durations into Morse symbols."""
    global decoded_text

    is_pressing   = False
    press_start   = 0.0
    last_release  = 0.0
    local_symbols = ""
    gap_checked   = False

    while not stop_event.is_set():
        space_down = keyboard.is_pressed("space")

        if space_down and not is_pressing:
            is_pressing   = True
            press_start   = time.time()
            gap_checked   = False

        elif not space_down and is_pressing:
            is_pressing   = False
            duration      = time.time() - press_start
            last_release  = time.time()
            symbol        = classify_signal(duration)
            local_symbols += symbol
            output_q.put(("symbol", symbol))

        # ── Gap detection ───────────────────────────────────
        if not is_pressing and last_release > 0:
            silence = time.time() - last_release

            if silence >= WORD_GAP and local_symbols and not gap_checked:
                letter = decode_morse(local_symbols)
                decoded_text  += letter + " "
                output_q.put(("letter", letter))
                output_q.put(("space",  None))
                speak_text(letter)
                local_symbols = ""
                gap_checked   = True

            elif silence >= LETTER_GAP and local_symbols and not gap_checked:
                letter = decode_morse(local_symbols)
                decoded_text  += letter
                output_q.put(("letter", letter))
                speak_text(letter)
                local_symbols = ""
                gap_checked   = True

        time.sleep(0.01)

    # Finalise any symbols still in the buffer when stopped
    if local_symbols:
        letter = decode_morse(local_symbols)
        decoded_text += letter
        output_q.put(("letter", letter))


# ── Main display loop ─────────────────────────────────────────
decoded_text = ""
raw_log      = ""

kb_thread = threading.Thread(target=_keyboard_input, args=(output_q, stop_event), daemon=True)
handle    = display(render_keyboard_html("", "", False), display_id=True)

print("⌨️  Keyboard mode active.")
print(f"   Short Space = dot  |  Long Space (>{DOT_DURATION}s) = dash")
print(f"   Pause {LETTER_GAP}s → letter  |  Pause {WORD_GAP}s → word space")
print("   Press ESC to stop.\n")

try:
    kb_thread.start()
    while True:
        if keyboard.is_pressed("esc"):
            break

        # Drain the event queue into the display log
        try:
            while True:
                kind, value = output_q.get_nowait()
                if kind == "symbol":
                    raw_log += value
                elif kind == "letter":
                    raw_log += f" <b>[{value}]</b> "
                elif kind == "space":
                    raw_log += "&nbsp;&nbsp;"
        except queue.Empty:
            pass

        handle.update(render_keyboard_html(
            raw_log,
            decoded_text,
            keyboard.is_pressed("space"),
        ))
        time.sleep(0.1)

finally:
    stop_event.set()
    kb_thread.join(timeout=2)
    handle.update(render_keyboard_html(raw_log, decoded_text, False))
    print(f"\n✅ Session ended.")
    print(f"   Decoded message: {decoded_text.strip() if decoded_text.strip() else '(nothing decoded)'}")

✅ Loaded tap model from tap_model.pkl



Press ESC to stop.



## 🎙️ Microphone Mode — ML-Filtered Tap Decoder

> **Requires calibration first** (run the Calibration cell above).

The listener loads your trained model and validates every sound against it before
recording a dot or dash — blocking out keyboard noise, voices, and other ambient sounds.

| Action | Meaning |
|--------|---------|
| Short tap | `.` dot |
| Long tap (> 0.15 s) | `−` dash |
| Pause ≥ 0.4 s | End of letter |
| Pause ≥ 1.0 s | End of word |
| `ESC` | Stop |

In [ ]:
# ── Load saved tap model ──────────────────────────────────────
MODEL_PATH = "tap_model.pkl"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at '{MODEL_PATH}'.\n"
        "Run the Calibration cell first to record taps and train the detector."
    )
with open(MODEL_PATH, "rb") as f:
    tap_model = pickle.load(f)
print(f"✅ Loaded tap model from {MODEL_PATH}\n")

# ── Shared state ──────────────────────────────────────────────
stop_event      = threading.Event()
output_q        = queue.Queue()
lock            = threading.Lock()

decoded_text    = ""
is_sound        = False
start_time      = 0.0
last_sound_time = 0.0
symbol_buffer   = ""
gap_checked     = False
current_volume  = 0.1
noise_floor     = 0.01
current_thresh  = 0.0
ml_label        = "—"
tap_buffer      = []


# ── Audio callback (runs per audio frame) ─────────────────────
def _audio_callback(indata, frames, time_info, status):
    global is_sound, start_time, last_sound_time, symbol_buffer, gap_checked
    global current_volume, noise_floor, current_thresh, ml_label, tap_buffer

    amplified      = indata * GAIN
    volume         = np.linalg.norm(amplified) / len(amplified) ** 0.5
    current_volume = volume

    # Adapt noise floor only during silence
    if not is_sound:
        noise_floor = NOISE_SMOOTH * noise_floor + (1 - NOISE_SMOOTH) * volume

    threshold      = noise_floor * SPIKE_FACTOR
    current_thresh = threshold

    with lock:
        if volume > threshold:
            if not is_sound:
                is_sound    = True
                start_time  = time.time()
                gap_checked = False
                tap_buffer  = [amplified.copy()]
            else:
                tap_buffer.append(amplified.copy())
            ml_label = "detecting…"

        else:
            if is_sound:
                # Tap just ended — validate against the ML model
                duration        = measure_duration(start_time)
                is_sound        = False
                last_sound_time = time.time()

                full_tap   = np.concatenate(tap_buffer).flatten()
                chunk_size = 1024
                if len(full_tap) >= chunk_size:
                    best_rms, best_chunk = 0, full_tap[:chunk_size]
                    for ci in range(0, len(full_tap) - chunk_size + 1, chunk_size // 2):
                        c = full_tap[ci:ci + chunk_size]
                        r = np.sqrt(np.mean(c ** 2))
                        if r > best_rms:
                            best_rms, best_chunk = r, c
                    feats = extract_features(best_chunk).reshape(1, -1)
                else:
                    feats = extract_features(full_tap).reshape(1, -1)

                tap_prob = tap_model.predict_proba(feats)[0][1]

                if tap_prob > 0.1:
                    symbol        = classify_signal(duration)
                    symbol_buffer += symbol
                    output_q.put(("symbol", symbol))
                    ml_label = f"TAP ({tap_prob:.0%})"
                else:
                    ml_label = f"rejected ({tap_prob:.0%})"
                tap_buffer = []
            else:
                ml_label = "quiet"


# ── Gap-monitoring thread ─────────────────────────────────────
def _group_symbols(output_q, stop_event):
    global symbol_buffer, decoded_text, gap_checked

    while not stop_event.is_set():
        time.sleep(0.05)
        with lock:
            if is_sound or last_sound_time == 0.0:
                continue
            silence = time.time() - last_sound_time

            if silence >= WORD_GAP and symbol_buffer and not gap_checked:
                letter = decode_morse(symbol_buffer)
                decoded_text  += letter + " "
                output_q.put(("letter", letter))
                output_q.put(("space",  None))
                speak_text(letter)
                symbol_buffer = ""
                gap_checked   = True

            elif silence >= LETTER_GAP and symbol_buffer and not gap_checked:
                letter = decode_morse(symbol_buffer)
                decoded_text  += letter
                output_q.put(("letter", letter))
                speak_text(letter)
                symbol_buffer = ""
                gap_checked   = True


# ── Main display loop ─────────────────────────────────────────
decoded_text = ""
raw_log      = ""

gap_thread = threading.Thread(target=_group_symbols, args=(output_q, stop_event), daemon=True)
handle     = display(render_mic_html("", "", 0.0, 0.01, 0.01, "—"), display_id=True)

print("🎙️  Microphone mode active.")
print("   Tap the table to send Morse code. Press ESC to stop.\n")

try:
    with sd.InputStream(callback=_audio_callback, channels=1, samplerate=SAMPLE_RATE):
        gap_thread.start()
        while True:
            if keyboard.is_pressed("esc"):
                break

            try:
                while True:
                    kind, value = output_q.get_nowait()
                    if kind == "symbol":
                        raw_log += value
                    elif kind == "letter":
                        raw_log += f" <b>[{value}]</b> "
                    elif kind == "space":
                        raw_log += "&nbsp;&nbsp;"
            except queue.Empty:
                pass

            handle.update(render_mic_html(
                raw_log, decoded_text,
                current_volume, current_thresh, noise_floor, ml_label,
            ))
            time.sleep(0.1)

finally:
    stop_event.set()
    gap_thread.join(timeout=1)
    if symbol_buffer:
        letter = decode_morse(symbol_buffer)
        decoded_text += letter
        raw_log      += f" <b>[{letter}]</b>"
    handle.update(render_mic_html(raw_log, decoded_text, 0.0, current_thresh, noise_floor, "—"))
    print(f"\n✅ Session ended.")
    print(f"   Decoded message: {decoded_text.strip() if decoded_text.strip() else '(nothing decoded)'}")

## 🔍 Mic Volume Diagnostic

Run this cell **before calibration** if mic mode isn't picking up your taps.
Tap the table several times during the 5-second recording to see how loud your taps are
versus the background noise, and whether the current `GAIN` / `SPIKE_FACTOR` settings will work.

In [ ]:
# Record 5 seconds and print volume statistics to help tune GAIN / SPIKE_FACTOR.
print(f"🎤  Recording 5 s (GAIN = {GAIN}×) — TAP THE TABLE NOW!\n")

recording = sd.rec(int(5 * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
recording = recording * GAIN

blocksize  = 1024
rms_values = np.array([
    np.linalg.norm(recording[i:i + blocksize]) / len(recording[i:i + blocksize]) ** 0.5
    for i in range(0, len(recording) - blocksize, blocksize)
])

mean_vol        = rms_values.mean()
max_vol         = rms_values.max()
adaptive_thresh = mean_vol * SPIKE_FACTOR

print(f"  Min  volume : {rms_values.min():.6f}")
print(f"  Max  volume : {max_vol:.6f}")
print(f"  Mean (noise): {mean_vol:.6f}")
print(f"  Threshold   : {adaptive_thresh:.6f}  (mean × {SPIKE_FACTOR})")
print(f"  Peak/Mean   : {max_vol / mean_vol:.1f}×")
print()

if max_vol > adaptive_thresh:
    print(f"  ✅ Taps ARE detectable — peaks are {max_vol / mean_vol:.1f}× above noise (need {SPIKE_FACTOR}×)")
else:
    print(f"  ⚠️  Taps NOT strong enough — only {max_vol / mean_vol:.1f}× above noise (need {SPIKE_FACTOR}×)")
    print(f"      Fix: increase GAIN  or  decrease SPIKE_FACTOR")

print()
print("  Volume timeline (each character ≈ 50 ms):")
print("  ", end="")
for v in rms_values[::2]:
    if v > adaptive_thresh:
        print("█", end="")
    elif v > adaptive_thresh * 0.5:
        print("▄", end="")
    else:
        print("░", end="")
print()